# Lab 16 — Distribution Shift & Generalization

*AI Cybersecurity Masterclass · Lab 16 · AI60secLab*

**Companion video:** [▶ Why Security AI Gets Dumber Over Time](https://www.youtube.com/watch?v=7BqRTjWFUBw&list=PLG4l6Ssi9A00)

**Research question:** How different are the prepared UNSW-NB15 training and test distributions, and can a classifier distinguish which dataset a row came from?

A domain-classifier AUC near 0.5 indicates the sampled domains are difficult to distinguish. Higher AUC means stronger detectable differences.

## Security foundation: every network is different

A detector trained on one network meets a different world on another: different applications, users, devices, and attackers. That is why commercial security tools need a **tuning period** after deployment, and why "it worked in the vendor's lab" is weak evidence.

The same applies between a benchmark and production. UNSW-NB15 was generated in a testbed, so a model's score on it says how well it learned *that* testbed. Measuring how different two datasets are (this lab's domain classifier) is a quick check of how far results might transfer.

In [1]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

Project root: /home/claude/run/exec
Data folder : /home/claude/run/exec/data/raw


In [2]:
import pandas as pd

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.drift import (
    normalized_median_shift,
    domain_classifier_auc,
)

train, test = load_unsw(DATA_DIR)

X_train, _ = split_xy(train, "label")
X_test, _ = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

In [3]:
numeric_columns = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

marginal_shift = normalized_median_shift(
    X_train,
    X_test,
    numeric_columns,
)

display(marginal_shift.head(20).round(4))

,feature,reference_median,current_median,reference_iqr,normalized_median_shift
16,swin,0.0000,2.550000e+02,2.550000e+02,1.0000
19,dwin,0.0000,2.550000e+02,2.550000e+02,1.0000
1,spkts,2.0000,6.000000e+00,1.000000e+01,0.4000
10,sloss,0.0000,1.000000e+00,3.000000e+00,0.3333
23,smean,73.0000,6.500000e+01,4.300000e+01,-0.1860
37,ct_srv_dst,4.0000,5.000000e+00,1.000000e+01,0.1000
3,sbytes,430.0000,5.340000e+02,1.304000e+03,0.0798
9,dload,1447.0227,2.112951e+03,2.784487e+04,0.0239
21,synack,0.0000,4.000000e-04,2.330000e-02,0.0190
0,dur,0.0016,1.410000e-02,6.681000e-01,0.0188


In [4]:
domain_auc, domain_model = domain_classifier_auc(
    X_train,
    X_test,
    sample_per_domain=20000,
    random_state=42,
)

print(
    "Domain-classifier validation ROC-AUC:",
    round(domain_auc, 4),
)

Domain-classifier validation ROC-AUC: 0.6692


In [5]:
from src.models import random_forest_pipeline
from src.evaluation import binary_metrics, binary_metrics_frame

y_train = train["label"]
y_test = test["label"]

security_model = random_forest_pipeline(X_train)
security_model.fit(X_train, y_train)

pred_train = security_model.predict(X_train)
score_train = security_model.predict_proba(X_train)[:, 1]

pred_test = security_model.predict(X_test)
score_test = security_model.predict_proba(X_test)[:, 1]

comparison = pd.concat([
    binary_metrics_frame(
        "Training distribution",
        binary_metrics(y_train, pred_train, score_train),
    ),
    binary_metrics_frame(
        "Held-out test distribution",
        binary_metrics(y_test, pred_test, score_test),
    ),
])

display(comparison.round(4))

,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,roc_auc
model,,,,,,,
Training distribution,0.9981,0.9991,0.9981,0.9986,0.0019,0.0019,1.0000
Held-out test distribution,0.8708,0.8163,0.9876,0.8938,0.2722,0.0124,0.9805


## Interpretation

Which features shifted most? How distinguishable are the train and test domains? Did security-model performance change across them?

**Takeaway:** unfamiliar data can appear without an adversary; generalization must be measured separately from clean training performance.

## From lab to SOC

Before trusting any detector on a new network, run it in alert-only mode, compare its alerts to what analysts confirm, and tune. Treat benchmark scores as a starting hypothesis, not a performance guarantee.

**Check yourself**
1. A domain classifier separates your training and production data easily. What should you expect from your detector?
2. Why do security vendors ask for a tuning period?
3. Which features are most likely to differ between a testbed and a hospital network?